# Notebook Structure
*Please Note: We strongly recommend you jump to the Inference Pipeline, as implementing the embedding experiments can be time consuming, especially downloding model weights. Embedded FAISS Vectorbase files needed in Inferece Pipeline are included in this codebase.* 

This notebook is organized as below:
1. **Chunking and Embedding**: We explored different chunking methods and embedding models:
- **Chunking**: Fixed-size Chunking, Semantic Chunking (TF-IDF Similarity, Embedding Similarity)
- **Embedding**: Qwen3-embedding-0.6B, Qwen3-embedding-4B

2. **Inference Pipeline**: We first demonstrate this pipeline by implementing on our pre-built Benchmark, which consists of 50 QA pairs about East Asian Cuisine. Input payload is the Benchmark by default, and can be modified in 1.1 Environmental Config by changing the value of `INPUT_FILE`.

3. **Evaluation Pipeline**: We implement evaluation on the output payload generated by the Inference Pipeline. Metrics involved in this part are shown below:
- **Retrival**: Recall@k, MRR@k, nDGC@k
- **Generation**: Token F1, Bert-Score



# 1. Chunking


## 1.1 Fixed Size Chunking
- **Strategy**: respect source record boundaries (each record = one section),
               merge tiny sections upward within the same title, then split
               oversized sections at sentence boundaries.


- **Chunk size**: target ~150 words, hard ceiling 200 words.

In [ ]:
from pathlib import Path
from chunking.chunker import run_fixed_chunking

FIXED_INPUT_PATH = Path("./corpus/corpus.jsonl")
FIXED_OUTPUT_PATH = Path("./artifacts/chunks/corpus_fixed.jsonl")
FIXED_TARGET_WORDS = 150
FIXED_MAX_WORDS = 200

fixed_chunks_path = run_fixed_chunking(
    input_path=FIXED_INPUT_PATH,
    output_path=FIXED_OUTPUT_PATH,
    target_words=FIXED_TARGET_WORDS,
    max_words=FIXED_MAX_WORDS,
)
print(f"Fixed chunks saved to: {fixed_chunks_path}")


Loading records from corpus/corpus.jsonl …
  1003 records loaded.
  147 (source, title) groups found.

Done.  1590 chunks written to artifacts/chunks/corpus_fixed.jsonl
  word count  min=6  max=200  mean=138  median=163
  chunks exceeding hard ceiling (200 w): 0

Sample chunks:
  [0] 12. Japan / ['Lead (part 1)'] (167 w) — Japan has the most Michelin star restaurants in the world, so there is something undeniably special about its cuisine. T…
  [1] 12. Japan / ['Lead (part 2)'] (83 w) — One thing I learned about Okinawa is that there is a small karaoke bar by the beach that will sell you cocktails for a c…
  [2] 12. Japan / ['Bento box (part 1)'] (182 w) — “Bento” derives from a word meaning “convenient” and it basically describes a variety of food, commonly including meat/f…
Fixed chunks saved to: artifacts/chunks/corpus_fixed.jsonl


## 1.2 TF-IDF Similarity
- **Strategy**:
Within each (source, title) group, compute TF-IDF cosine similarity between
every pair of adjacent chunks.  Merge consecutive chunks whose similarity
exceeds threshold — provided the merged result stays within max_words.

In [ ]:
from pathlib import Path
from chunking.semantic_chunker_tfidf import run_tfidf_chunking
TFIDF_INPUT_PATH = FIXED_OUTPUT_PATH
TFIDF_OUTPUT_PATH = Path("./artifacts/chunks/chunks_tfidf.jsonl")
TFIDF_THRESHOLD = 0.10
TFIDF_MAX_WORDS = 260
TFIDF_MIN_WORDS = 40

tfidf_chunks_path = run_tfidf_chunking(
    input_path=TFIDF_INPUT_PATH,
    output_path=TFIDF_OUTPUT_PATH,
    threshold=TFIDF_THRESHOLD,
    max_words=TFIDF_MAX_WORDS,
    min_words=TFIDF_MIN_WORDS,
)
print(f"TF-IDF chunks saved to: {tfidf_chunks_path}")

Loading chunks from artifacts/chunks/corpus_fixed.jsonl …
  1590 input chunks.
Building TF-IDF matrix …
  Matrix shape: (1590, 15000)
  147 (source, title) groups.
  70 recipe groups (part-only merging).
Post-processing: splitting oversized chunks and absorbing stubs …
  1374 → 1345 chunks after post-processing.

Done.  1345 chunks → artifacts/chunks/chunks_tfidf.jsonl
  Threshold: 0.1  |  max_words: 260
  Word count  min=31  max=260  mean=163  median=182
  Merge method breakdown:
    none           :  1120 chunks
    tfidf          :   107 chunks
    stub           :    80 chunks
    post_stub      :    24 chunks
    same_section   :    14 chunks
  Avg originals per merged chunk: 2.09

Sample output:
  [0] 12. Japan / ['Lead (part 1)'] (167w, merged_from=[0]) — Japan has the most Michelin star restaurants in the world, so there is something undeniably special …
  [1] 12. Japan / ['Lead (part 2)'] (83w, merged_from=[1]) — One thing I learned about Okinawa is that there is a small karao

## 1.3 Embedding Similarity
- **Strategy**:
Encode every chunk with a lightweight sentence-transformer model.
Within each (source, title) group, merge consecutive chunks whose
embedding cosine similarity exceeds threshold — provided the merged
result stays within max_words.

In [61]:
from pathlib import Path
from chunking.semantic_chunker_embedding import run_embedding_chunking

EMBED_INPUT_PATH = FIXED_OUTPUT_PATH
EMBED_OUTPUT_PATH = Path("./artifacts/chunks/chunks_embedding.jsonl")
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"
EMBED_THRESHOLD = 0.78
EMBED_MAX_WORDS = 260
EMBED_MIN_WORDS = 40
EMBED_BATCH_SIZE = 64
EMBED_RUN_CALIBRATE = False

embedding_chunks_path = run_embedding_chunking(
    input_path=EMBED_INPUT_PATH,
    output_path=EMBED_OUTPUT_PATH,
    model_name=EMBED_MODEL_NAME,
    threshold=EMBED_THRESHOLD,
    max_words=EMBED_MAX_WORDS,
    min_words=EMBED_MIN_WORDS,
    batch_size=EMBED_BATCH_SIZE,
    run_calibrate=EMBED_RUN_CALIBRATE,
)

print(f"Embedding chunks saved to: {embedding_chunks_path}")

Loading chunks from artifacts/chunks/corpus_fixed.jsonl …
  1590 input chunks.
Loading model: BAAI/bge-small-en-v1.5 …


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding 1590 chunks in batches of 64 …


Batches:   0%|          | 0/25 [00:00<?, ?it/s]

  Encoding done in 3.0s. Shape: (1590, 384)

Processing 147 title groups …
Post-processing: splitting oversized chunks and absorbing stubs …
  1433 → 1404 chunks after post-processing.

Done.  1404 chunks → artifacts/chunks/chunks_embedding.jsonl
  Model: BAAI/bge-small-en-v1.5  |  Threshold: 0.78  |  max_words: 260
  Word count  min=31  max=260  mean=157  median=178
  Merge method breakdown:
    none           :  1236 chunks
    stub           :    83 chunks
    embedding      :    47 chunks
    post_stub      :    24 chunks
    same_section   :    14 chunks
  Avg boundary sim of merged chunks: 0.839

Sample output:
  [0] 12. Japan / ['Lead (part 1)', 'Lead (part 2)'] (250w, avg_sim=0.803) — Japan has the most Michelin star restaurants in the world, so there is something undeniably special …
  [1] 12. Japan / ['Bento box (part 1)'] (182w, no merge) — “Bento” derives from a word meaning “convenient” and it basically describes a variety of food, commo…
  [2] 12. Japan / ['Bento box (par

# 2. Chunk Embedding


We implemented cross experiments among differnt Chunking methods and Embedding models.
- **Embedding Models**: Qwen3-embedding-0.6B, Qwen3-embedding-4B

Meanwhile, we implemented a **Natural Language Prefix** strategy: To prepend a short human-readable context header (which is combined with title and section of the chunk) to each text chunk before encoding. A comparison study with Non-Prefixed (baseline) method shows Prefixing can enhance the retrieval performance.

In [74]:
RUN_CHUNK_EMBEDDING = False  # set to True to run chunk embedding

from pathlib import Path
from embedding.embed_chunks import run_chunk_embedding

EMBED_CHUNKING_METHOD = "embedding"  # "fixed" | "tfidf" | "embedding"

EMBED_INPUT_PATH = {
    "fixed": Path("./artifacts/chunks/corpus_fixed.jsonl"),
    "tfidf": Path("./artifacts/chunks/chunks_tfidf.jsonl"),
    "embedding": Path("./artifacts/chunks/chunks_embedding.jsonl"),
}[EMBED_CHUNKING_METHOD]

EMBED_MODEL_REF = "Qwen/Qwen3-Embedding-0.6B"   # or "Qwen/Qwen3-Embedding-4B"
EMBED_OUTPUT_DIR = Path("./artifacts/faiss")
EMBED_INDEX_TYPE = "FlatIP"                      
EMBED_NORMALIZE = True
EMBED_BATCH_SIZE = 4
EMBED_MAX_LENGTH = 512
EMBED_DRY_RUN = False

if RUN_CHUNK_EMBEDDING:
    artifact_paths = run_chunk_embedding(
        input_path=EMBED_INPUT_PATH,
        output_dir=EMBED_OUTPUT_DIR,
        model_ref=EMBED_MODEL_REF,
        index_type=EMBED_INDEX_TYPE,
        normalize=EMBED_NORMALIZE,
        batch_size=EMBED_BATCH_SIZE,
        max_length=EMBED_MAX_LENGTH,
        dry_run=EMBED_DRY_RUN,
    )
    print("Embedding artifacts:")
    for k, v in artifact_paths.items():
        print(f"  {k}: {v}")

# 3. Inference Pipeline




## 3.1 Environment Config

In [1]:
import sys, json
from pathlib import Path

#  project root on path 
ROOT = Path("./").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

BENCHMARK = ROOT / "benchmark/benchmark.json"
INPUT_FILE = BENCHMARK
# INPUT_FILE  = ROOT / "input_payload.json"
OUTPUT_FILE = ROOT / "output_payload.json"   # saved in project root

print(f"Project root: {ROOT}")

Project root: /home/arkkuma/final


## 3.2 FAISS Vectorbase Path Config

In [2]:
FAISS_MANIFEST = ROOT / "artifacts/faiss/chunks_embedding.Qwen__Qwen3-Embedding-0.6B.nlp_prefix.FlatIP.manifest.json"
META_JSONL     = ROOT / "artifacts/faiss/chunks_embedding.Qwen__Qwen3-Embedding-0.6B.nlp_prefix.FlatIP.meta.jsonl"

## 3.3 Retrieve and Rerank

**Strategy:** BM25 (sparse) + Dense (FAISS) → RRF fusion → Cross-encoder rerank


In [3]:
#  retrieval hyper-parameters 
BM25_TOP_K      = 30   # BM25 candidates per query
DENSE_TOP_K     = 30   # Dense candidates per query
RRF_K           = 20    # RRF smoothing constant
RRF_WEIGHTS     = [1,0, 1.0]   # [bm25_weight, dense_weight]

#  cross-encoder rerank hyper-parameters 
RERANK_INPUT_K  = 10    # candidates fed to cross-encoder
FINAL_TOP_K     = 5     # results returned per query
# HuggingFace ID or local path
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

In [4]:
from retrieval.pipeline import RetrievalPipeline
# Initialize the inference pipeline.
pipeline = RetrievalPipeline(
    meta_jsonl_path     = META_JSONL,
    faiss_manifest_path = FAISS_MANIFEST,
    cross_encoder_model = CROSS_ENCODER_MODEL,
    bm25_top_k          = BM25_TOP_K,
    dense_top_k         = DENSE_TOP_K,
    rrf_k               = RRF_K,
    rrf_weights         = RRF_WEIGHTS,
    rerank_input_k      = RERANK_INPUT_K,
    final_top_k         = FINAL_TOP_K,
)
print("Pipeline is ready.")

[BM25] Indexed 1404 docs from chunks_embedding.Qwen__Qwen3-Embedding-0.6B.nlp_prefix.FlatIP.meta.jsonl
[Dense] FAISS loaded: 1404 vectors, dim=1024 from chunks_embedding.Qwen__Qwen3-Embedding-0.6B.nlp_prefix.FlatIP.index
[Reranker] Using HuggingFace: cross-encoder/ms-marco-MiniLM-L-6-v2
[Reranker] Loading cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pipeline is ready.


## 3.4 RAG with Retrieved Context

In [5]:
# Generation Config 

# HuggingFace ID or local path
GENERATOR_MODEL    = "Qwen/Qwen2.5-0.5B-Instruct"

MAX_NEW_TOKENS     = 256    # token budget per response
CONTEXT_MAX_CHARS  = 2000   # character cap for the context block in the prompt

#  System Prompt — edit freely 
SYSTEM_PROMPT = """\
You are a knowledgeable culinary assistant. \
Answer the user's question using only the information provided in the context. \
Be concise and factual. \
If the context does not contain enough information to answer, say so clearly \
and do not make up details.\
"""

In [6]:
from generation.generator import RAGGenerator

generator = RAGGenerator(
    model_ref         = GENERATOR_MODEL,
    max_new_tokens    = MAX_NEW_TOKENS,
    context_max_chars = CONTEXT_MAX_CHARS,
)

[Generator] Using HuggingFace: Qwen/Qwen2.5-0.5B-Instruct
[Generator] Loading model on cuda: Qwen/Qwen2.5-0.5B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[Generator] Ready.


In [7]:
#  RAG Pipeline 
with open(INPUT_FILE, encoding="utf-8") as f:
    input_payload = json.load(f)

# Step 1: retrieve context for every query
retrieval_output = pipeline.run_batch(input_payload)

# Step 2: fill in responses
final_output = generator.fill_responses(
    pipeline_output = retrieval_output,
    system_prompt   = SYSTEM_PROMPT,
)

# Step 3: save to project root
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(final_output, f, indent=2, ensure_ascii=False)
print(f"Output payload is saved to: {OUTPUT_FILE}")

# print("\n Final Output Payload ")
# print(json.dumps(final_output, indent=2, ensure_ascii=False))

[Dense] Local model not found, using HuggingFace: Qwen/Qwen3-Embedding-0.6B
[Dense] Loading query encoder: Qwen/Qwen3-Embedding-0.6B


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  Generating [1/50] What cooking technique is used to make tempura?...
  Generating [2/50] What is chicken teriyaki?...
  Generating [3/50] What is dashi traditionally made with?...
  Generating [4/50] Where did takoyaki originate?...
  Generating [5/50] What is inside a takoyaki?...
  Generating [6/50] What is xiǎochī in Taiwanese cuisine?...
  Generating [7/50] Where is the Taiwanese oyster omelette thought to have origi...
  Generating [8/50] What gives the Taiwanese oyster omelette its thick consisten...
  Generating [9/50] What principle do northern Chinese apply to food preparation...
  Generating [10/50] Why is hot pot considered a healthy meal?...
  Generating [11/50] What does 'yum cha' mean in Cantonese?...
  Generating [12/50] What is a buuz typically filled with?...
  Generating [13/50] Why is rice not grown locally in Mongolia?...
  Generating [14/50] What type of rice is required to make onigiri?...
  Generating [15/50] What happens if spicy miso udon soup reaches a boil?

# 4. Evaluation Pipeline

## 4.1 Retrieval Evaluation

In [14]:
from evaluation.metrics import evaluate_from_files, print_metrics

# OUTPUT_FILE = ROOT / "output_payload.json"
# BENCHMARK = ROOT / "benchmark" / "benchmark.json"

print(f"Retrieval output: {OUTPUT_FILE}")
print(f"Benchmark: {BENCHMARK}")

EVAL_KS         = [3, 5]   # cut-offs for Recall@k and NDCG@k
EVAL_VERBOSE    = False    

metrics = evaluate_from_files(
    output_path=OUTPUT_FILE,
    benchmark_path=BENCHMARK,
    ks=EVAL_KS,
    verbose=EVAL_VERBOSE,
)

print_metrics(metrics)

Retrieval output: /home/arkkuma/final/output_payload.json
Benchmark: /home/arkkuma/final/benchmark/benchmark.json

  Metric              Score
----------------------------------------
  MRR                0.0157
  Recall@3           0.0200
  NDCG@3             0.0100
  Recall@5           0.0600
  NDCG@5             0.0264


## 4.2 Generation Quality Evaluation

Evaluates responses in `output_payload.json` against benchmark ground-truth answers.

| Metric | Description |
|--------|-------------|
| **Token F1** | Unigram overlap (SQuAD-style): precision / recall / F1 after lowercasing + punctuation removal |
| **BERTScore** | Contextual embedding similarity via `roberta-large` |

In [ ]:
from evaluation.gen_metrics import evaluate_generation, print_gen_metrics
from evaluation.metrics import load_benchmark

# Load output
OUTPUT_FILE = ROOT / "output_payload.json"

#  Generation Evaluation Config 
# BERTScore backbone: "roberta-large" (best quality) or "distilbert-base-uncased" (faster on CPU)
BERTSCORE_MODEL    = "distilbert-base-uncased"
GEN_EVAL_VERBOSE   = False

with open(OUTPUT_FILE, encoding="utf-8") as f:
    output_payload = json.load(f)

# Load benchmark (supports both old and new schema automatically)
eval_benchmark = load_benchmark(BENCHMARK)
print(f"Output items  : {len(output_payload['results'])}")
print(f"Benchmark items: {len(eval_benchmark)}")

NameError: name 'BENCHMARK' is not defined

In [15]:
# fallback defaults if Config cell was not re-run
_bertscore_model = globals().get("BERTSCORE_MODEL", "distilbert-base-uncased")
_gen_verbose     = globals().get("GEN_EVAL_VERBOSE", False)

gen_results = evaluate_generation(
    output_payload  = output_payload,
    benchmark       = eval_benchmark,
    bertscore_model = _bertscore_model,
    verbose         = _gen_verbose,
)

print_gen_metrics(gen_results)

NameError: name 'evaluate_generation' is not defined

In [ ]:
#  Per-query breakdown (worst → best by Token F1) 
per_item = sorted(gen_results["per_item"], key=lambda x: x["token_f1"]["f1"])

print(f"{'#':>3}  {'Token F1':>9}  {'BERT-F1':>9}  Query / Prediction vs Reference")
print("-" * 100)
for item in per_item:
    tf1 = item["token_f1"]["f1"]
    bf1 = item["bert_score"]["f1"]
    q   = item["query"][:55]
    print(f"     {tf1:>9.4f}  {bf1:>9.4f}  Q: {q}")
    print(f"     {'':9}  {'':9}  P: {item['prediction'][:90]}")
    print(f"     {'':9}  {'':9}  R: {item['reference'][:90]}")
    print()

NameError: name 'gen_results' is not defined